# HDS Data Ingestion Statistics

The `HDSDataIngestionStatistics` class is designed to provide various utility functions for analyzing and comparing data in the HDS medallion lakehouse architecture. 
It allows users to collect statistics from raw NDJSON source data, bronze ClinicalFHIR tables, and silver Lakehouse tables. Additionally, it provides methods to compare data across different layers (source, bronze, and silver) to identify discrepancies, ensuring data consistency and integrity.

## Pre-reqs

Kindly ensure that you attach a lakehouse on this notebook. Preferably attach the bronze lakehouse

### Key Functions:
1. **raw_source_ndjson_data_stats**: Computes statistics for NDJSON files within a root directory.
2. **get_bronze_clinical_fhir_statistics**: Computes statistics on the number of records and distinct ids per resource type in the Bronze ClinicalFHIR Delta table.
3. **get_lakehouse_statistics**: Computes the statistics of all tables in specified Silver Lakehouse databases.
4. **compare_clinical_fhir_to_lakehouse**: Compares ClinicalFHIR statistics to Lakehouse statistics to identify discrepancies.
5. **compare_raw_source_to_bronze**: Compares statistics between raw NDJSON source data and the Bronze ClinicalFHIR table to identify discrepancies.

### Note on Required Parameters
Please ensure that you update the following parameters with the correct paths and database names for your environment:

- **`root_source_data_path`**: The root path where the raw NDJSON source data is located.
- **`bronze_delta_table_path`**: The path to the Bronze ClinicalFHIR Delta table.
- **`databases`**: A list of database names to analyze in the Lakehouse.

In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""

In [ ]:
root_source_data_path = f'abfss://{workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Files/SampleData/Clinical/FHIR-NDJSON/FHIR-HDS/51KSyntheticPatients'
bronze_delta_table_path = f"abfss://{workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/ClinicalFhir"
databases = ["healthcare1_msft_silver"] 

# HDSDataIngestionStatistics Class

In [ ]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor

class HDSDataIngestionStatistics:
    def __init__(self, spark):
        self.spark = spark

    def raw_source_ndjson_data_stats(self, root_source_data_path) -> DataFrame:
        """
        Function to get statistics on the number of records in NDJSON files within the root source path, recursively.
        
        :param root_source_data_path: Root path to source data.
        :return: A DataFrame containing resourceType and count of records per resourceType.
        """
        # Use Spark to recursively read all NDJSON files from the root source path
        df = self.spark.read.format("json").option("recursiveFileLookup", "true").load(root_source_data_path)
        df = df.filter(input_file_name().endswith('.ndjson'))
        
        # Count the number of records per resource type
        resource_counts_df = df.groupBy("resourceType").agg(
            count("*").alias("record_count")
        )
        
        # Calculate total rows and print summary
        total_rows = df.count()
        total_files = len(df.select(input_file_name()).distinct().collect())
        
        print(f'Total number of records across all NDJSON files: {total_rows}')
        print(f'Total number of NDJSON files in sample data: {total_files}')
        
        print('\nResource Type Counts:')
        resource_counts = resource_counts_df.collect()
        for row in resource_counts:
            print(f'{row["resourceType"]}: {row["record_count"]}')
        
        return resource_counts_df

    def get_bronze_clinical_fhir_statistics(self, delta_table_path) -> DataFrame:
        """
        Function to get statistics on the number of records per resourceType and the number of distinct ids per resourceType.
        
        :param delta_table_path: Path to the Delta table.
        :return: A DataFrame containing resourceType, total count, and distinct id count.
        """
        # Read the entire Delta table
        df = self.spark.read.format("delta").load(delta_table_path)
        
        # Group by resourceType and calculate counts and distinct counts
        bronze_stats_df = df.groupBy(col("resourceType")).agg(
            count("*").alias("count"),
            countDistinct("id").alias("distinct_count")
        )
        
        return bronze_stats_df

    def get_lakehouse_statistics(self, databases) -> DataFrame:
        """
        Function to get statistics on the number of records in each table within a list of databases.
        
        :param databases: A list of database names to analyze.
        :return: A DataFrame containing database name, table name, and record count.
        """
        # Define schema explicitly to avoid empty schema issues
        schema = StructType([
            StructField("database", StringType(), True),
            StructField("resource", StringType(), True),
            StructField("silver_record_count", LongType(), True)
        ])
        
        # Initialize an empty list to hold the results
        stats_list = []

        def process_table(database, table):
            try:
                if table.tableType == 'MANAGED':  # Ensure it's a table and not a view or temporary table
                    table_name = table.name
                    # Load the table and count the number of records using DataFrame API, minimizing shuffles
                    record_count = self.spark.read.table(f"{database}.{table_name}").count()
                    # Append the result to the stats list
                    stats_list.append((database, table_name, record_count))
            except Exception as e:
                print(f"Error processing table {table.name} in database {database}: {str(e)}")

        for database in databases:
            try:
                # Set the current database
                self.spark.catalog.setCurrentDatabase(database)
                
                # Get a list of tables in the current database
                tables = self.spark.catalog.listTables(database)
                
                # Process tables in parallel
                with ThreadPoolExecutor() as executor:
                    for table in tables:
                        executor.submit(process_table, database, table)

            except Exception as e:
                print(f"Error processing database {database}: {str(e)}")

        # Handle the case where stats_list is empty
        if not stats_list:
            print("No tables found or no records available.")
            return None

        # Convert the results to a DataFrame
        lakehouse_stats_df = self.spark.createDataFrame(stats_list, schema)
        
        # Order the DataFrame by record count in descending order
        lakehouse_stats_df = lakehouse_stats_df.orderBy(col("silver_record_count").desc())
        
        return lakehouse_stats_df

    def compare_clinical_fhir_to_lakehouse(self, clinical_fhir_df, lakehouse_stats_df) -> DataFrame:
        """
        Function to compare the distinct count of records in the ClinicalFHIR table with the count of records
        in each of the tables in the lakehouse.
        
        :param clinical_fhir_df: DataFrame containing ClinicalFHIR statistics.
        :param lakehouse_stats_df: DataFrame containing Lakehouse statistics.
        :return: A DataFrame containing resourceType, ClinicalFHIR distinct count, Silver record count, total count in ClinicalFHIR, and the difference.
        """
        # Join the DataFrames on resourceType
        comparison_df = clinical_fhir_df.join(
            lakehouse_stats_df, 
            clinical_fhir_df.resourceType == lakehouse_stats_df.resource, 
            "outer"
        ).select(
            coalesce(clinical_fhir_df.resourceType, lakehouse_stats_df.resource).alias("resource"), 
            coalesce(col("distinct_count"), lit(0)).alias("clinical_fhir_distinct_count"), 
            coalesce(col("count"), lit(0)).alias("clinical_fhir_total_count"),
            coalesce(col("silver_record_count"), lit(0)).alias("silver_record_count"),
            (coalesce(col("distinct_count"), lit(0)) - coalesce(col("silver_record_count"), lit(0))).alias("discrepancy_count")
        )
        
        # Check for discrepancies and print details if any
        discrepancies_df = comparison_df.filter(col("discrepancy_count") != 0)
        if discrepancies_df.count() > 0:
            print("Discrepancies found between ClinicalFHIR and Lakehouse tables:")
            display(discrepancies_df)
        else:
            print("No discrepancies found between ClinicalFHIR and Lakehouse tables.")

        # Order the DataFrame by record count in descending order
        comparison_df = comparison_df.orderBy(col("clinical_fhir_distinct_count").desc(), col("silver_record_count").desc())
        return comparison_df

    def compare_raw_source_to_bronze(self, raw_source_df, bronze_df) -> DataFrame:
        """
        Function to compare the distinct count of records in the raw source NDJSON data with the count of records
        in the bronze ClinicalFHIR table.
        
        :param raw_source_df: DataFrame containing raw source NDJSON statistics.
        :param bronze_df: DataFrame containing Bronze ClinicalFHIR statistics.
        :return: A DataFrame containing resourceType, raw source record count, bronze record count, and the difference.
        """
        # Join the DataFrames on resourceType
        comparison_df = raw_source_df.join(
            bronze_df, 
            raw_source_df.resourceType == bronze_df.resourceType, 
            "outer"
        ).select(
            coalesce(raw_source_df.resourceType, bronze_df.resourceType).alias("resource"),
            coalesce(col("record_count"), lit(0)).alias("raw_source_record_count"),
            coalesce(col("count"), lit(0)).alias("bronze_record_count"),
            (coalesce(col("record_count"), lit(0)) - coalesce(col("count"), lit(0))).alias("discrepancy_count")
        )
        
        # Check for discrepancies and print details if any
        discrepancies_df = comparison_df.filter(col("discrepancy_count") != 0)
        if discrepancies_df.count() > 0:
            print("Discrepancies found between raw source NDJSON data and Bronze ClinicalFHIR table:")
            display(discrepancies_df)
        else:
            print("No discrepancies found between raw source NDJSON data and Bronze ClinicalFHIR table.")
        
        return comparison_df

**Initialize the HDSDataIngestionStatitisticsClass**

In [ ]:
stats = HDSDataIngestionStatistics(spark)

**Get raw source NDJSON data stats**

- Computes statistics from the raw NDJSON files, including record counts per resource type and the total number of files.

In [ ]:
# Get raw source NDJSON data stats
source_data_stats_df = stats.raw_source_ndjson_data_stats(root_source_data_path).cache()

**Get bronze table statistics**
   - Analyzes the Bronze ClinicalFHIR table to compute statistics on the number of records and distinct `id`s per resource type.

In [ ]:
# Get bronze table statistics
bronze_statistics_df = stats.get_bronze_clinical_fhir_statistics(bronze_delta_table_path).cache()
display(bronze_statistics_df)

**Get lakehouse statistics for silver tables**
   - Retrieves statistics for each table in specified Silver Lakehouse databases, including the total count of records per table.

In [ ]:
# Get lakehouse statistics for silver tables
lakehouse_stats_df = stats.get_lakehouse_statistics(databases).cache()
display(lakehouse_stats_df)

**Compare Raw Source Data to Bronze**
   - Compares the statistics between the raw source data and the Bronze ClinicalFHIR table to identify discrepancies in record counts per resource type

**NOTE** - One potential reason for discrepancies here could be related to files that failed validation and therefore the data in those files were not ingested

In [ ]:
# Compare Raw Source to Bronze
source_to_bronze_comparison_df = stats.compare_raw_source_to_bronze(source_data_stats_df, bronze_statistics_df)
display(source_to_bronze_comparison_df)

**Compare Bronze ClinicalFHIR to Silver lakeshouse tables**

- Compares the statistics between the Bronze ClinicalFHIR table and the Silver Lakehouse tables to identify discrepancies in record counts per resource type

**NOTE**: _DocumentReferenceContent_ is expected to have discrepancies since its a child resource of _DocumentReference_. Therefore the source data in Bronze is _DocumentReference_



In [ ]:
# Compare ClinicalFHIR to Lakehouse
comparison_df = stats.compare_clinical_fhir_to_lakehouse(bronze_statistics_df, lakehouse_stats_df)
display(comparison_df)

In [ ]:
source_data_stats_df.unpersist()
bronze_statistics_df.unpersist()
lakehouse_stats_df.unpersist()